In [14]:
import pandas as pd
import json
from datetime import date
from constants import DATA_PATH, TMP_PATH, PUB_PATH
import subprocess
from utils import check_and_create_directory

## Copy files from temporary to publication directory

In [15]:
check_and_create_directory(PUB_PATH)

In [16]:
# prepare commands
commands = [
    f"cp {TMP_PATH}manuscripts.csv {PUB_PATH}manuscripts.csv",
    f"cp {TMP_PATH}verses.csv {PUB_PATH}verses.csv",
    f"cp {TMP_PATH}words.csv {PUB_PATH}words.csv",
    f"cp {TMP_PATH}occurrences.csv {PUB_PATH}occurrences.csv",
    f"jq -s -c '.' {DATA_PATH}parsed/errors/*.json > {PUB_PATH}parsing-errors.json",
]

# Run commands
for command in commands:
    subprocess.run(command, shell=True, check=True)

## Read files for data manipulation

In [17]:
# read files from output directory
manuscripts_df = pd.read_csv(
    PUB_PATH + "manuscripts.csv",
    low_memory=False,
)

verses_df = pd.read_csv(
    PUB_PATH + "verses.csv",
    low_memory=False,
)

words_df = pd.read_csv(
    PUB_PATH + "words.csv",
    low_memory=False,
)

occurrences_df = pd.read_csv(
    PUB_PATH + "occurrences.csv",
    low_memory=False,
)

### Filter for 'name' type words

As SemDH2024 dataset is only for words of type name, we filter for that type and write the results to two seperate files:
- `name_words.csv`
- `name_occurrences.csv`

In [27]:
# get only names from words dataframe
names_words_df = words_df[words_df["type"] == "name"]
# names_words_df.drop(columns=["type"], inplace=True)
# get the unique variantIDs of all names
names_wordIDs = names_words_df["wordID"].unique()
# filter occurrences by variantIDs of names
names_occurrences_df = occurrences_df[occurrences_df["wordID"].isin(names_wordIDs)]

# set column types
names_words_df.astype(
    {
        "label:en": "string",
        "gender": "string",
        "label:el:norm": "string",
        "factgrid": "string",
        "variant": "string",
        "wordID": "Int64",
        "variantID": "Int64",
    }
)
# set NA values
names_words_df = names_words_df.fillna("NA")

# set column types
names_occurrences_df.astype(
    {
        "verse_id": "string",
        "variantID": "Int64",
        "wordID": "Int64",
        "occurrence": "boolean",
    }
)

# write both dataframes to files
names_words_df.to_csv(PUB_PATH + "name_words.csv", index=False)
names_occurrences_df.to_csv(PUB_PATH + "name_occurrences.csv", index=False)

# Create Data Dicts for each file

In [19]:
NA_VALUE = "NA"
NAN_VALUE = "-1"
TITLE = "A Corpus of Biblical Names in the Greek New Testament to Study the Additions, Omissions, and Variations across different Manuscripts"
FILE_FORMAT = "csv"
CONTENT_URL = "https://github.com/chr-werner/SemDH2024-GreekNewTestamentNames"
DATE_PUBLISHED = str(date.today())
KEYWORDS = ["New Testament", "Biblical Names", "Textual Variation Units"]
LICENCE = "CC BY 4.0"
FUNDER = "Deutsche Forschungsgemeinschaft (DFG, German Research Foundation) 513300936"
AUTHORS = [
    {
        "identifier": "0009-0008-9907-251X",
        "givenName": "Christoph",
        "familyName": "Werner",
        "email": "christoph.werner@hs-wismar.de",
        "affiliation": "University of Wismar",
    },
    {
        "identifier": "0000-0002-9784-7034",
        "givenName": "Zacharias",
        "familyName": "Shoukry",
        "email": "zacharias.shoukry@uni-rostock.de",
        "affiliation": "University of Rostock",
    },
    {
        "identifier": "0000-0003-1098-208X",
        "givenName": "Soham",
        "familyName": "Al-Suadi",
        "email": "soham.al-suadi@uni-rostock.de",
        "affiliation": "University of Rostock",
    },
    {
        "identifier": "0000-0002-7925-3363",
        "givenName": "Frank",
        "familyName": "Krüger",
        "email": "frank.krueger@hs-wismar.de",
        "affiliation": "University of Wismar",
    },
]

In [20]:
def generate_json(
    name: str,
    file_format: str,
    file_name: str,
    content_url: str,
    date_published: str,
    keywords: list[str],
    license_info: str,
    funder: str,
    authors: list[dict[str, str]],
    distribution_name: str,
    variable_measured: list[dict[str, str]],
    output_file: str,
):
    """Builder function for JSON data dictionary

    :param name: Title of the work
    :param file_format: Data file format
    :param file_name: Data file name
    :param content_url: GitHub content url
    :param date_published: Data publishign date
    :param keywords: List of keywords
    :param license_info: Licence to be applied on the data
    :param funder: Funder of the project
    :param authors: List of authors of the project
    :param distribution_name: Distribution title (same as parameter name)
    :param variable_measured: List of variables measured
    :param output_file: File path to output file
    :return:
    """
    data = {
        "@context": "https://schema.org/",
        "@type": "Dataset",
        "name": name,
        "fileFormat": file_format,
        "fileName": file_name,
        "contentUrl": content_url,
        "datePublished": date_published,
        "keywords": keywords,
        "license": license_info,
        "funder": funder,
        "author": [
            {
                "@type": "Person",
                "identifier": author["identifier"],
                "givenName": author["givenName"],
                "familyName": author["familyName"],
                "email": author["email"],
                "affiliation": author["affiliation"],
            }
            for author in authors
        ],
        "distribution": {
            "@type": "DataDownload",
            "name": distribution_name,
            "fileFormat": file_format,
        },
        "variableMeasured": [
            {
                "@type": "PropertyValue",
                "identifier": variable["identifier"],
                "unitText": variable["unitText"],
                "disambiguatingDescription": {
                    "@type": "Text",
                    "missingValuesAllowed": variable["missingValuesAllowed"],
                    "missingValuesValues": variable.get("missingValuesValues", "NA"),
                },
                "description": variable["description"],
                # "sameAs": variable["sameAs"],
            }
            for variable in variable_measured
        ],
    }

    # Save to a file
    with open(output_file, "w") as f:
        f.write(json.dumps(data, indent=4))

In [21]:
bkv = {
    "identifier": "bkv",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Verse Identifier by BKV Scheme",
}
century = {
    "identifier": "century",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Verse Identifier by BKV Scheme",
}
dbpedia = {
    "identifier": "dbpedia",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "DBpedia item id",
}
docID = {
    "identifier": "docID",
    "unitText": "integer",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Document Identifier by INTF scheme",
}
edition_date = {
    "identifier": "edition_date",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Transcription date",
}
edition_version = {
    "identifier": "edition_version",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Transcription edition",
}
encoding_version = {
    "identifier": "encoding_version",
    "unitText": "numeric",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Version of encoding scheme",
}
factgrid = {
    "identifier": "factgrid",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "FactGrid ItemID",
}
funder = {
    "identifier": "funder",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Founding institution",
}
ga = {
    "identifier": "ga",
    "unitText": "character",
    "missingValuesAllowed": False,
    "description": "Document Identifier by Gregory Aland Scheme",
    # "sameAs": "https://www.wikidata.org/prop/direct/P1577"
}
gender = {
    "identifier": "gender",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Genus",
}
label = {
    "identifier": "label",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Manuscript name",
}
label_el_norm = {
    "identifier": "label:el:norm",
    "unitText": "character",
    "missingValuesAllowed": False,
    "description": "Normalized greek label",
}
label_en = {
    "identifier": "label:en",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "English label",
}
lection = {
    "identifier": "lection",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Lection Identifier",
}
nkv = {
    "identifier": "nkv",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Verse Identifier by NKV Scheme",
}
occurrence = {
    "identifier": "occurrence",
    "unitText": "logical",
    "missingValuesAllowed": False,
    "description": "Indicator of occurrence",
}
pagesCount = {
    "identifier": "pagesCount",
    "unitText": "integer",
    "missingValuesAllowed": True,
    "missingValuesValues": NAN_VALUE,
    "description": "Number of pages",
}
leavesCount = {
    "identifier": "leavesCount",
    "unitText": "integer",
    "missingValuesAllowed": True,
    "missingValuesValues": NAN_VALUE,
    "description": "Number of leaves",
}
publisher = {
    "identifier": "publisher",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Transcription publisher",
}
publishing_date = {
    "identifier": "publishing_date",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Transcription publishing date",
}
source = {
    "identifier": "source",
    "unitText": "character",
    "missingValuesAllowed": False,
    "description": "Source of data",
}
sponsor = {
    "identifier": "sponsor",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Transcription sponsor",
}
text = {
    "identifier": "text",
    "unitText": "character",
    "missingValuesAllowed": False,
    "description": "Transcription without gap annotations",
}
transcript = {
    "identifier": "transcript",
    "unitText": "character",
    "missingValuesAllowed": False,
    "description": "Transcription with gap annotations",
}
witness = {
    "identifier": "witness",
    "unitText": "character",
    "missingValuesAllowed": True,
    "missingValuesValues": NA_VALUE,
    "description": "Witness (scribe) who wrote or did corrections",
}
word_type = {
    "identifier": "type",
    "unitText": "character",
    "missingValuesAllowed": False,
    # TODO: can either be name, profession, nominal, group
    "description": "Word type",
}
variant = {
    "identifier": "variant",
    "unitText": "character",
    "missingValuesAllowed": False,
    "description": "Spelling variant",
}
variantID_words = {
    "identifier": "variantID",
    "unitText": "integer",
    "missingValuesAllowed": False,
    "description": "Unique variant identifier",
}
variantID_occurrences = {
    "identifier": "variantID",
    "unitText": "integer",
    "missingValuesAllowed": True,
    "missingValuesValues": NAN_VALUE,
    "description": "Unique variant identifier",
}
verse_id = {
    "identifier": "verse_id",
    "unitText": "character",
    "missingValuesAllowed": False,
    "description": "Unique verse identifier",
}
wordID = {
    "identifier": "wordID",
    "unitText": "integer",
    "missingValuesAllowed": False,
    "description": "Unique word identifier",
}

In [22]:
# Generate WORDS JSON
generate_json(
    name=TITLE,
    file_format=FILE_FORMAT,
    file_name="words",
    content_url=CONTENT_URL,
    date_published=DATE_PUBLISHED,
    keywords=KEYWORDS,
    license_info=LICENCE,
    funder=FUNDER,
    authors=AUTHORS,
    distribution_name=TITLE,
    variable_measured=[
        label_en,
        label_el_norm,
        gender,
        factgrid,
        variant,
        word_type,
        wordID,
        variantID_words,
    ],
    output_file=PUB_PATH + "words.json",
)

In [23]:
# Generate VERSES JSON
generate_json(
    name=TITLE,
    file_format=FILE_FORMAT,
    file_name="verses",
    content_url=CONTENT_URL,
    date_published=DATE_PUBLISHED,
    keywords=KEYWORDS,
    license_info=LICENCE,
    funder=FUNDER,
    authors=AUTHORS,
    distribution_name=TITLE,
    variable_measured=[
        bkv,
        edition_date,
        edition_version,
        encoding_version,
        funder,
        ga,
        lection,
        nkv,
        publisher,
        publishing_date,
        source,
        sponsor,
        transcript,
        text,
        verse_id,
        witness,
    ],
    output_file=PUB_PATH + "verses.json",
)

In [24]:
# Generate OCCURRENCES JSON
generate_json(
    name=TITLE,
    file_format=FILE_FORMAT,
    file_name="occurrences",
    content_url=CONTENT_URL,
    date_published=DATE_PUBLISHED,
    keywords=KEYWORDS,
    license_info=LICENCE,
    funder=FUNDER,
    authors=AUTHORS,
    distribution_name=TITLE,
    variable_measured=[
        variantID_occurrences,
        occurrence,
        wordID,
        verse_id,
    ],
    output_file=PUB_PATH + "occurrences.json",
)

In [25]:
# Generate MANUSCRIPTS JSON
generate_json(
    name=TITLE,
    file_format=FILE_FORMAT,
    file_name="manuscripts",
    content_url=CONTENT_URL,
    date_published=DATE_PUBLISHED,
    keywords=KEYWORDS,
    license_info=LICENCE,
    funder=FUNDER,
    authors=AUTHORS,
    distribution_name=TITLE,
    variable_measured=[
        docID,
        pagesCount,
        leavesCount,
        ga,
        century,
        source,
        label,
        dbpedia,
    ],
    output_file=PUB_PATH + "manuscripts.json",
)